# BOQ Similarity Retrieval · Fine-Tune Qwen3-Embedding-0.6B (v36 Production)

**Pipeline:** Drive/Upload PDFs → Smart Anchors → Content Clustering → MNRL Pairs → Fine-Tune → Retrieve

### v36 Fixes (over v35)
| # | Issue | Fix |
|---|-------|-----|
| 1 | Cell 10 searched for `adapter_config.json` in checkpoints — SentenceTransformers never creates this file → merge always skipped → PEFT weights saved with `lora_A/lora_B/base_layer` keys → UNEXPECTED/MISSING on reload | Removed checkpoint adapter loading entirely. Merge the in-memory model directly with `merge_and_unload()` — it already has trained LoRA weights after `trainer.train()` |
| 2 | No assertion after merge → silent failure possible | Added assertion: `assert not lora_params` — hard fail if merge didn't work |
| 3 | Stale PEFT artifacts (`adapter_config.json` etc.) could remain in output dir | Added post-save cleanup to remove any stale PEFT files |
| 4 | Verification reload failure was silently caught | Changed to `raise` on verification failure |

### v32 Fixes (over v31)
| # | Issue | Fix |
|---|-------|-----|
| 1 | `sys` not imported → `NameError` on Cell 2 startup | Added `import sys` |
| 2 | LoRA merge used `isinstance(PeftModel)` → skipped for `PeftModelForFeatureExtraction` → saved PEFT weights → UNEXPECTED/MISSING keys on reload | Changed to `hasattr(model, 'merge_and_unload')` — always works |
| 3 | No Qwen3 instruction prefix → model treats query and corpus identically → poor retrieval discrimination (Dam BOQ retrieving Hospital, Solar Farm) | Added `QUERY_INSTRUCTION` to anchor side of training pairs AND to query at inference |
| 4 | `doc_keywords` not saved to `cluster_state.json` → lost on kernel restart → degraded anchor quality for new BOQs | Added to state; restored in Cell 11 and EVAL-A |
| 5 | Anchor scoring used `.mean(axis=1)` → dilutes discriminative signal across vocabulary | Changed to `.sum(axis=1)` — rewards lines with most domain-specific terms |
| 6 | Old checkpoints auto-resumed without version check | Added `RESUME_TRAINING` flag (default False) |
| 7 | EVAL-A didn't restore `doc_keywords` | Fixed in EVAL-A state restore block |
| 8 | `CKPT_LIMIT=20` wasted Drive space | Reduced to 3 |


In [ ]:
# ============================================================
# CELL 0 · Mount Google Drive
# Run FIRST — all checkpoints and artefacts go to Drive.
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
DRIVE_BASE = '/content/drive/MyDrive/BOQ_FineTune_Final'
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/best_model',  exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/cluster_state', exist_ok=True)
print(f'✓ Drive mounted.')
print(f'  Checkpoints  → {DRIVE_BASE}/checkpoints')
print(f'  Best model   → {DRIVE_BASE}/best_model')
print(f'  Cluster state→ {DRIVE_BASE}/cluster_state')


In [ ]:
# ============================================================
# CELL 1: Install Dependencies  (run once, then restart runtime)
# ============================================================
import subprocess, sys, os, re

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip"] + list(args), check=True)

# ── Step 0: Fix Pillow core/package version mismatch ─────────────────────
# Colab has a split Pillow install that pip cannot fully overwrite:
#   _imaging.so  → 11.3.0  (compiled C extension, write-protected)
#   __init__.py  → 12.2.0  (Python files, also sometimes write-protected)
# Strategy: read the core version from _imaging.so, then patch
# PIL/__init__.py to report the same version so PIL.Image's version
# guard passes. No reinstall needed.
print("[Step 0] Fixing Pillow core/package version mismatch...")

# 1. Get the version baked into _imaging.so
try:
    import _imaging
    core_ver = getattr(_imaging, 'PILLOW_VERSION', None)
except Exception:
    core_ver = None

if not core_ver:
    # Fallback: grep the .so binary for the version string
    import glob
    sos = glob.glob('/usr/local/lib/python3.*/dist-packages/PIL/_imaging*.so')
    for so in sos:
        try:
            raw = open(so, 'rb').read()
            m = re.search(rb'(\d+\.\d+\.\d+)', raw[raw.find(b'PILLOW'):raw.find(b'PILLOW')+30])
            if m: core_ver = m.group(1).decode(); break
        except Exception:
            pass

if not core_ver:
    core_ver = '11.3.0'  # known Colab default
print(f"  _imaging.so version: {core_ver}")

# 2. Find PIL/__init__.py and patch the __version__ string
import glob
init_files = glob.glob('/usr/local/lib/python3.*/dist-packages/PIL/__init__.py')
patched = False
for init_path in init_files:
    src = open(init_path).read()
    cur_ver = re.search(r'__version__\s*=\s*"([^"]+)"', src)
    if cur_ver and cur_ver.group(1) != core_ver:
        print(f"  Patching {init_path}: {cur_ver.group(1)} → {core_ver}")
        new_src = re.sub(
            r'(__version__\s*=\s*")([^"]+)(")',
            lambda m: m.group(1) + core_ver + m.group(3),
            src
        )
        open(init_path, 'w').write(new_src)
        patched = True
    elif cur_ver:
        print(f"  {init_path} already at {cur_ver.group(1)} — no patch needed.")

# 3. Also patch PIL/Image.py to downgrade its __version__ reference if needed
img_files = glob.glob('/usr/local/lib/python3.*/dist-packages/PIL/Image.py')
for img_path in img_files:
    src = open(img_path).read()
    # The version check block: replace the raise with a warning
    if 'raise ImportError(msg)' in src and 'Core version:' in src:
        new_src = src.replace(
            'raise ImportError(msg)',
            '# raise ImportError(msg)  # patched: version mismatch tolerated'
        )
        open(img_path, 'w').write(new_src)
        print(f"  Patched version-check raise→pass in {img_path}")

# 4. Purge PIL from sys.modules so patches take effect on next import
import sys as _sys
for _k in [k for k in _sys.modules if k == 'PIL' or k.startswith('PIL.')]:
    _sys.modules.pop(_k, None)

# 5. Quick verify
import PIL, PIL.Image
print(f"  PIL loaded OK — reported version: {PIL.__version__}")

# ── Step 1: torchao ───────────────────────────────────────────────────────
print("[Step 1] Upgrading torchao>=0.16.0 ...")
pip("install", "-q", "torchao>=0.16.0")

# ── Step 2: Core packages ─────────────────────────────────────────────────
pkgs = [
    "packaging>=24.0", "wheel",
    "sentence-transformers>=3.3.0",
    "transformers>=4.51.0",
    "accelerate>=0.30.0",
    "datasets>=2.19.0",
    "pdfplumber>=0.10.0",
    "pymupdf>=1.23.0",
    "easyocr",
    "pdf2image",
    "scikit-learn>=1.3.0",
    "bitsandbytes>=0.43.0",
    "huggingface_hub",
    "rank_bm25",
    "peft>=0.11.0",
    "tqdm",
]
print("[Step 2] Installing remaining packages ...")
pip("install", "-U", *pkgs)

# ── Step 3: System packages ───────────────────────────────────────────────
print("[Step 3] Installing poppler-utils ...")
subprocess.run(["apt-get", "update", "-qq"], check=False)
try:
    subprocess.run(["apt-get", "install", "-y", "poppler-utils"], check=True)
except subprocess.CalledProcessError as e:
    print(f"⚠️  apt-get failed ({e.returncode}).")

print("\n✅ All dependencies installed.")
print("⚠️  CRITICAL: Runtime > Restart runtime — then start from Cell 0.")


In [ ]:
from pathlib import Path

# ============================================================
# CELL 3 · Configuration — edit only this cell
# ============================================================

# ── Model ────────────────────────────────────
MODEL_NAME  = "Qwen/Qwen3-Embedding-0.6B"
MAX_SEQ_LEN = 1024

QUERY_INSTRUCTION = (
    "Instruct: Given a Bill of Quantities (BOQ) document from the construction "
    "domain, retrieve the most semantically similar BOQ documents that describe "
    "the same type of construction project.\nQuery: "
)

# ── Training ─────────────────────────────────
BATCH_SIZE    = 8
GRAD_ACCUM    = 4
EPOCHS        = 1
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1

# ── Memory / Stability ──────────────────────────────
FP16          = True
BF16          = False
GRAD_CKPT     = True
USE_8BIT_ADAM = True

# ── Checkpointing ───────────────────────────────
SAVE_STEPS    = 100
EVAL_STEPS    = 100
CKPT_LIMIT    = 3

# ── Clustering ────────────────────────
# FIX: Increased to 0.70 to force separate clusters for different project types
SIMILARITY_THRESHOLD = 0.70
MIN_CLUSTER_SIZE     = 2

# ── TF-IDF Boilerplate Filter ─────────────────────
MAX_DF = 0.35

# ── Loss + Data ───────────────────────────
GRAD_CACHE_MINI_BATCH = 2
TRIPLETS_PER_DOC      = 5
TRIPLET_MARGIN        = 0.3
MAX_ANCHOR_CHARS      = 2000
MIN_DESC_LEN          = 10
VAL_RATIO             = 0.20
TEST_RATIO            = 0.10

# ── Paths ───────────────────────────────
PDF_DIR    = Path("/content/boq_pdfs")
DRIVE_BASE = "/content/drive/MyDrive/BOQ_FineTune_Neww"
OUTPUT_DIR = Path(f"{DRIVE_BASE}/best_model")
CKPT_DIR   = Path(f"{DRIVE_BASE}/checkpoints")
EMBED_CACHE= Path(f"{DRIVE_BASE}/corpus_embeddings.pt")
CLUSTER_STATE_DIR = Path(f"{DRIVE_BASE}/cluster_state")

for d in [PDF_DIR, OUTPUT_DIR, CKPT_DIR, CLUSTER_STATE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Configuration loaded.")

In [ ]:
# ============================================================
# CELL 4 · Load PDFs — from Drive folder OR upload a ZIP
# ============================================================
from google.colab import files as colab_files
import zipfile

DRIVE_CORPUS = Path(f"{DRIVE_BASE}/corpus_pdfs/BOQ")

drive_pdfs = (
    sorted(list(DRIVE_CORPUS.rglob("*.pdf")) + list(DRIVE_CORPUS.rglob("*.PDF")))
    if DRIVE_CORPUS.exists() else []
)

if drive_pdfs:
    print(f"Found {len(drive_pdfs)} PDFs in Drive corpus.")
    _dup = 0
    for p in tqdm(drive_pdfs, desc="Copying from Drive"):
        dest = PDF_DIR / p.name
        if dest.exists():
            for _i in range(2, 9999):
                cand = PDF_DIR / f"{dest.stem}_{_i}{dest.suffix}"
                if not cand.exists():
                    dest = cand; _dup += 1; break
        shutil.copy2(str(p), str(dest))
    if _dup:
        print(f"  ⚠️  {_dup} duplicate filenames auto-renamed.")
else:
    print("No Drive corpus found — upload a ZIP of BOQ PDFs.")
    uploaded = colab_files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(PDF_DIR)

pdf_files = sorted(list(PDF_DIR.rglob("*.pdf")) + list(PDF_DIR.rglob("*.PDF")))
print(f"\n{len(pdf_files)} PDFs ready in {PDF_DIR}")
for p in pdf_files[:8]:
    print(f"   {p.name}  ({p.stat().st_size/1024:.1f} KB)")
if len(pdf_files) > 8:
    print(f"   ... and {len(pdf_files)-8} more")
assert len(pdf_files) >= 2, "Need ≥ 2 PDFs to build training pairs."


In [ ]:
# ============================================================
# CELL 5 · OCR Pipeline  (pdfplumber → PyMuPDF → EasyOCR)
# ============================================================
import gc
import numpy as np
import pdfplumber
import fitz # PyMuPDF
import torch
import shutil
from tqdm.auto import tqdm
from typing import List, Dict

def _total_chars(pages: List[Dict]) -> int:
    return sum(len(p.get("text", "")) for p in pages)


def extract_pdfplumber(pdf_path: str) -> List[Dict]:
    pages = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text(x_tolerance=3, y_tolerance=3) or ""
                if len(text.strip()) < 100:
                    tables = page.extract_tables(
                        {"vertical_strategy": "lines_strict",
                         "horizontal_strategy": "lines_strict"}) or []
                    rows = []
                    for tbl in tables:
                        for row in (tbl or []):
                            cells = [str(c or "").strip() for c in (row or [])]
                            line  = "  ".join(c for c in cells if c)
                            if line: rows.append(line)
                    if rows: text = "\n".join(rows)
                if text.strip():
                    pages.append({"page": i+1, "text": text.strip()})
    except Exception:
        pass
    return pages


def extract_pymupdf(pdf_path: str) -> List[Dict]:
    pages = []
    try:
        doc = fitz.open(pdf_path)
        for i, page in enumerate(doc):
            blocks = page.get_text("blocks", sort=True)
            texts  = [b[4].strip() for b in blocks
                      if isinstance(b[4], str) and b[4].strip()]
            text   = "\n".join(texts)
            if text: pages.append({"page": i+1, "text": text})
        doc.close()
    except Exception:
        pass
    return pages


_easyocr_vram_used = False

def extract_easyocr(pdf_path: str) -> List[Dict]:
    global _easyocr_vram_used
    pages = []
    try:
        import easyocr
        from pdf2image import convert_from_path
        if not hasattr(extract_easyocr, "_reader"):
            print("    Initialising EasyOCR (one-time) ...")
            extract_easyocr._reader = easyocr.Reader(
                ["en"], gpu=torch.cuda.is_available(), verbose=False)
            _easyocr_vram_used = torch.cuda.is_available()
        images = convert_from_path(pdf_path, dpi=200)
        for i, img in enumerate(images):
            result = extract_easyocr._reader.readtext(
                np.array(img), detail=0, paragraph=True)
            text = "\n".join(str(r) for r in result)
            if text.strip(): pages.append({"page": i+1, "text": text.strip()})
    except Exception as e:
        print(f"    EasyOCR error: {e}")
    return pages


def release_easyocr_vram():
    global _easyocr_vram_used
    if hasattr(extract_easyocr, "_reader"):
        del extract_easyocr._reader
    _easyocr_vram_used = False
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("EasyOCR VRAM released.")


def smart_extract(pdf_path: str, verbose: bool = False) -> str:
    name  = Path(pdf_path).name
    pages = extract_pdfplumber(pdf_path)
    if _total_chars(pages) >= 300:
        if verbose: print(f"  [pdfplumber] {name}")
        return "\n".join(p["text"] for p in pages)
    pages = extract_pymupdf(pdf_path)
    if _total_chars(pages) >= 300:
        if verbose: print(f"  [PyMuPDF]    {name}")
        return "\n".join(p["text"] for p in pages)
    if verbose: print(f"  [EasyOCR]    {name}")
    pages = extract_easyocr(pdf_path)
    return "\n".join(p["text"] for p in pages)


print("✓ OCR pipeline defined (pdfplumber → PyMuPDF → EasyOCR)")

In [ ]:
# ============================================================
# CELL 6 · Extract Descriptions from All PDFs
# ============================================================

_SKIP_PAT = re.compile(
    r'^(s\.?no\.?|description|unit|qty|quantity|rate|amount|'
    r'sub[-\s]?total|^total\b|^grand[-\s]?total\b|'
    r'bill\s+of\s+quantities|client\s*:|location\s*:|'
    r'ref\s*:|currency\s*:|page\s*\d)',
    re.IGNORECASE
)

def extract_boq_descriptions(raw_text: str, min_len: int = MIN_DESC_LEN) -> List[str]:
    descriptions, seen = [], set()
    for line in raw_text.splitlines():
        line = line.strip()
        if len(line) < min_len: continue
        alpha = sum(c.isalpha() for c in line)
        if alpha < 3: continue      # skip pure numbers/symbols, but keep short acronyms
        if _SKIP_PAT.match(line): continue
        key = " ".join(line.lower().split())        # normalised dedup key
        if key in seen: continue
        seen.add(key)
        descriptions.append(line)
    return descriptions

print(f"Extracting text from {len(pdf_files)} PDFs ...")
print("(pdfplumber → PyMuPDF → EasyOCR auto-chain)\n")

doc_texts:        Dict[str, str]       = {}
doc_descriptions: Dict[str, List[str]] = {}
failed = []

for pdf_path in tqdm(pdf_files, desc="Extracting"):
    stem     = pdf_path.stem
    raw_text = smart_extract(str(pdf_path), verbose=False)
    if len(raw_text.strip()) < 50:
        print(f"  ⚠️  Skipping {pdf_path.name} — no text")
        failed.append(str(pdf_path)); continue
    descs = extract_boq_descriptions(raw_text)
    if not descs:
        print(f"  ⚠️  Skipping {pdf_path.name} — no descriptions")
        failed.append(str(pdf_path)); continue
    doc_texts[stem]        = raw_text
    doc_descriptions[stem] = descs

print(f"\n✓ Extracted : {len(doc_texts):,} / {len(pdf_files)} PDFs")
if failed:
    print(f"  ⚠️  Failed ({len(failed)}): "
          + ", ".join(Path(f).name for f in failed[:5]))

counts = [len(d) for d in doc_descriptions.values()]
if counts:
    print(f"\nDescription stats  min/avg/max : "
          f"{min(counts)} / {int(np.mean(counts))} / {max(counts)}")

assert len(doc_texts) >= 2, "Need ≥ 2 successfully parsed PDFs!"
print("\n✓ Extraction complete.")


In [ ]:
import sys
import subprocess

def pip_install_temp(*args):
    subprocess.run([sys.executable, "-m", "pip"] + list(args), check=True, capture_output=True)

# Ensure Pillow is at a compatible version before importing libraries that depend on it
print("Ensuring consistent Pillow installation...")
try:
    # Uninstall any existing Pillow versions
    pip_install_temp("uninstall", "-y", "Pillow")
except subprocess.CalledProcessError:
    pass # Ignore if Pillow is not installed

# Install the version targeted by Cell 1's patch, which includes _Ink
pip_install_temp("install", "Pillow==11.3.0", "--force-reinstall")

# Purge PIL from sys.modules to ensure the newly installed version is loaded
for _k in [k for k in sys.modules if k == 'PIL' or k.startswith('PIL.')]:
    sys.modules.pop(_k, None)
print("Pillow re-installed to 11.3.0.")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict, Counter
from typing import List, Dict, Optional
from datasets import Dataset
import random
import pickle
import json
import numpy as np
import gc
import torch
from sentence_transformers import SentenceTransformer, util as st_util
from tqdm.auto import tqdm

# FIX: Define the missing anchor generation function
def make_smart_anchor(stem: str, descs: List[str]) -> str:
    """Creates a discriminative anchor text by combining the document name and top keywords."""
    # Use global doc_keywords and MAX_ANCHOR_CHARS from config
    kws = doc_keywords.get(stem, [])[:10]
    prefix = f"Project: {stem.replace('_', ' ')}\nKeywords: {', '.join(kws)}\n"
    body = "\n".join(descs[:15])
    max_len = globals().get('MAX_ANCHOR_CHARS', 2000)
    return (prefix + body)[:max_len]

# ============================================================
# CELL 7 · Hybrid Semantic Clustering + Structured Anchors + Triplets
# ============================================================

# ══════════════════════════════════════════════════════════════
# STEP 1: TF-IDF for Keyword Extraction
# ══════════════════════════════════════════════════════════════
print(f"STEP 1: TF-IDF keyword extraction (max_df={MAX_DF}) ...")
stems_list = list(doc_descriptions.keys())
all_texts  = [" ".join(doc_descriptions[s]) for s in stems_list]

doc_tfidf = TfidfVectorizer(
    stop_words   = "english",
    max_features = 20_000,
    max_df       = MAX_DF,
    min_df       = 2,
    token_pattern= r"(?u)\b(?=[a-zA-Z0-9]*[a-zA-Z])[a-zA-Z0-9]{3,}\b",
    sublinear_tf = True,
)
doc_tfidf_matrix = doc_tfidf.fit_transform(all_texts)
feature_names    = doc_tfidf.get_feature_names_out()

_dense_matrix = doc_tfidf_matrix.toarray()
doc_keywords = {}
doc_display_types = {}
for i, stem in enumerate(stems_list):
    vec = _dense_matrix[i]
    top_idx = np.argsort(-vec)[:20]
    kws = [feature_names[j] for j in top_idx if vec[j] > 0]
    doc_keywords[stem] = kws
    doc_display_types[stem] = " | ".join(kws[:3]) if kws else stem

del _dense_matrix; gc.collect()

# ══════════════════════════════════════════════════════════════
# STEP 2: Base Qwen3 Semantic Clustering
# ══════════════════════════════════════════════
print("STEP 2: Base model semantic clustering ...")
release_easyocr_vram()

_base_st = SentenceTransformer(MODEL_NAME, trust_remote_code=True)
_base_st.max_seq_length = MAX_SEQ_LEN
_doc_texts_for_cluster = [" ".join(doc_descriptions[s])[:MAX_SEQ_LEN*4] for s in stems_list]

base_embeddings = _base_st.encode(_doc_texts_for_cluster, batch_size=16, show_progress_bar=True, normalize_embeddings=True)
base_embeddings = np.array(base_embeddings, dtype=np.float32)
del _base_st; gc.collect(); torch.cuda.empty_cache()

dist_matrix = cosine_distances(base_embeddings).astype(np.float64)
clusterer = AgglomerativeClustering(n_clusters=None, distance_threshold=1.0 - SIMILARITY_THRESHOLD, metric="precomputed", linkage="average")
cluster_labels_arr = clusterer.fit_predict(dist_matrix)
base_sim_matrix = base_embeddings @ base_embeddings.T

# Group members
cluster_members = defaultdict(list)
for i, stem in enumerate(stems_list):
    cid = int(cluster_labels_arr[i])
    cluster_members[cid].append(stem)

# Map doc to its cluster ID/name
doc_types = {stem: int(cluster_labels_arr[i]) for i, stem in enumerate(stems_list)}
print(f"  Generated {len(cluster_members)} clusters.")

# ════════════════════
# STEP 3-5: Anchors, Split, Triplets
# ════════════════════
print("STEP 3-5: Anchors and Triplet Mining ...")
doc_anchors = {stem: make_smart_anchor(stem, doc_descriptions[stem]) for stem in stems_list}

# Split
random.shuffle(stems_list)
train_docs = stems_list[:int(len(stems_list)*0.8)]
val_docs   = stems_list[int(len(stems_list)*0.8):]

# Mining with fallback
def build_triplets(source_stems):
    triplets = []
    stem_to_idx_local = {s: i for i, s in enumerate(stems_list)}
    for anchor_stem in source_stems:
        a_cid = doc_types[anchor_stem]
        pos_pool = [s for s in cluster_members[a_cid] if s != anchor_stem]
        if not pos_pool: continue

        neg_pool = [s for s in source_stems if doc_types[s] != a_cid]
        if not neg_pool: neg_pool = [s for s in source_stems if s != anchor_stem]

        a_idx = stem_to_idx_local[anchor_stem]
        scored_pos = sorted(pos_pool, key=lambda s: base_sim_matrix[a_idx][stem_to_idx_local[s]], reverse=True)

        for pos_s in scored_pos[:TRIPLETS_PER_DOC]:
            neg_s = random.choice(neg_pool)
            triplets.append({"anchor": QUERY_INSTRUCTION + doc_anchors[anchor_stem], "positive": doc_anchors[pos_s]})
    return triplets

t_train = build_triplets(train_docs)
t_val   = build_triplets(val_docs)

train_ds = Dataset.from_dict({"anchor": [t["anchor"] for t in t_train], "positive": [t["positive"] for t in t_train]})
val_ds = Dataset.from_dict({"anchor": [t["anchor"] for t in t_val], "positive": [t["positive"] for t in t_val]})
print(f"  Final: {len(train_ds)} train / {len(val_ds)} val triplets.")

In [ ]:
# ============================================================
# CELL 8 · Load Qwen3-Embedding-0.6B + LoRA Injection
# ============================================================
from peft import get_peft_model, LoraConfig, PeftModel, PeftModelForFeatureExtraction
from sentence_transformers import SentenceTransformer, models as st_models
import torch
import gc

# Release EasyOCR VRAM before loading the embedding model
release_easyocr_vram()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU free before model load: {free_gb:.2f} GB")

print(f"\nLoading base model: {MODEL_NAME}")

# 1. Load Transformer
word_embedding_model = st_models.Transformer(
    MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    model_kwargs   = {"torch_dtype": torch.float32, "trust_remote_code": True},
)
# Qwen3-Embedding requires LEFT padding (decoder-based model)
word_embedding_model.tokenizer.padding_side = "left"

# 2. Add Pooling (Qwen3-Embedding uses LAST-TOKEN pooling)
pooling_model = st_models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_lasttoken    = True,
    pooling_mode_mean_tokens  = False,
    pooling_mode_cls_token    = False,
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
model.max_seq_length = MAX_SEQ_LEN

# 3. LoRA injection
print("Injecting LoRA adapters ...")
lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout   = 0.05,
    bias           = "none",
)

base_auto_model = model[0].auto_model
peft_model_raw  = get_peft_model(base_auto_model, lora_config)

if not hasattr(peft_model_raw, 'merge_and_unload'):
    print("  get_peft_model didn't wrap — wrapping manually ...")
    peft_model_raw = PeftModelForFeatureExtraction(base_auto_model, lora_config)

model[0].auto_model = peft_model_raw

if GRAD_CKPT:
    try:
        peft_model_raw.base_model.model.gradient_checkpointing_enable()
        peft_model_raw.enable_input_require_grads()
        print("  Gradient checkpointing enabled.")
    except Exception as e:
        print(f"  WARNING: Gradient checkpointing failed: {e}")

# Smoke test
test_emb = model.encode([QUERY_INSTRUCTION + "Test"], show_progress_bar=False)
print(f"  Embedding dim : {test_emb.shape[-1]}")
print("\n✓ Model ready for fine-tuning.")
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ============================================================
# CELL 9 · Fine-Tuning
# ============================================================
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
import torch
import json
from pathlib import Path

# Set RESUME_TRAINING=True to resume from last checkpoint on Drive
RESUME_TRAINING = False
_DISABLE_EVAL = False

loss_fn = CachedMultipleNegativesRankingLoss(
    model,
    mini_batch_size=GRAD_CACHE_MINI_BATCH
)
print(f"Loss : CachedMultipleNegativesRankingLoss (mini_batch={GRAD_CACHE_MINI_BATCH})")

optim = "adamw_torch"
if USE_8BIT_ADAM:
    try:
        import bitsandbytes
        optim = "adamw_bnb_8bit"
        print("Optimizer: AdamW 8-bit (bitsandbytes)")
    except ImportError:
        print("WARNING: bitsandbytes not found, using standard AdamW")

steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_RATIO))
est_min         = total_steps * 1.5 / 60

# Auto-adjust for small datasets
_save_steps = SAVE_STEPS
_eval_steps = EVAL_STEPS
if steps_per_epoch < SAVE_STEPS:
    _save_steps = max(1, steps_per_epoch)
    _eval_steps = _save_steps
    print(f"  Auto-adjusted SAVE/EVAL_STEPS -> {_save_steps}")

_eval_strategy = "no" if _DISABLE_EVAL else "steps"

training_args = SentenceTransformerTrainingArguments(
    output_dir                  = str(CKPT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = 1.0,
    warmup_steps                = warmup_steps,
    optim                       = optim,
    lr_scheduler_type           = "cosine",
    fp16                        = FP16,
    bf16                        = BF16,
    save_strategy               = "steps",
    save_steps                  = _save_steps,
    save_total_limit            = CKPT_LIMIT,
    eval_strategy               = _eval_strategy,
    eval_steps                  = _eval_steps if not _DISABLE_EVAL else None,
    load_best_model_at_end      = False,
    dataloader_num_workers      = 0,
    dataloader_drop_last        = True,
    logging_steps               = 10,
    report_to                   = "none",
)

trainer = SentenceTransformerTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = val_ds if not _DISABLE_EVAL else None,
    loss          = loss_fn,
)

print("\n🚀 Starting training ...")
trainer.train()
print("\n✓ Training complete!")

In [ ]:
# ============================================================
# CELL 10 · Merge LoRA Weights + Save Clean Model to Drive
#
# v36 FIX: The in-memory model already has trained LoRA weights
# after Cell 9 training. We merge directly — NO need to search
# for adapter_config.json in checkpoints (SentenceTransformers
# doesn't save separate adapter files in checkpoints).
#
# After merge_and_unload() the SentenceTransformer[0].auto_model
# is a plain HuggingFace model with NO LoRA tensors.
# save_pretrained() then writes standard weight files.
# ============================================================
import gc
import torch
import numpy as np
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
import os

print(f"Saving to: {OUTPUT_DIR}\n")

# ── 1. Log which checkpoint was best (informational only) ─────
best_ckpt_dir = None
if hasattr(trainer, 'state') and trainer.state.best_model_checkpoint:
    best_ckpt_dir = Path(trainer.state.best_model_checkpoint)
    print(f"  Trainer best checkpoint: {best_ckpt_dir}")
else:
    import glob as _glob
    ckpt_dirs = sorted(_glob.glob(str(CKPT_DIR / 'checkpoint-*')))
    if ckpt_dirs:
        best_ckpt_dir = Path(ckpt_dirs[-1])
        print(f"  Latest checkpoint: {best_ckpt_dir}")

# ── 2. Direct in-memory LoRA merge ────────────────────────────
# The model[0].auto_model is ALREADY the PeftModel with trained
# LoRA weights from Cell 9. Just merge and unload.
auto_model = model[0].auto_model
print(f"  auto_model type: {type(auto_model).__name__}")

if hasattr(auto_model, 'merge_and_unload'):
    print("  Merging LoRA weights into base model ...")
    merged_model = auto_model.merge_and_unload()
    model[0].auto_model = merged_model
    print(f"  Type after merge: {type(merged_model).__name__}")

    # Verify: no LoRA params remain
    lora_params = [n for n, _ in merged_model.named_parameters()
                   if 'lora' in n.lower()]
    if lora_params:
        print(f"  ❌ MERGE FAILED: {len(lora_params)} LoRA params still present!")
        for p in lora_params[:5]:
            print(f"     {p}")
        raise RuntimeError(f"LoRA merge failed — {len(lora_params)} LoRA params remain")
    else:
        print(f"  ✓ Zero LoRA parameters in merged model — merge successful.")
else:
    raise RuntimeError(
        "auto_model has no merge_and_unload() — "
        "PEFT wrapper not applied? Check Cell 8.")

# ── 3. Save merged SentenceTransformer ────────────────────────
model.save_pretrained(str(OUTPUT_DIR))
print(f"  SentenceTransformer saved → {OUTPUT_DIR}")

# Save tokenizer explicitly
try:
    model[0].tokenizer.save_pretrained(str(OUTPUT_DIR))
    print("  Tokenizer saved.")
except Exception as e:
    print(f"  WARNING: Tokenizer save: {e}")

# ── 4. Remove any stale PEFT adapter artifacts ────────────────
for _artifact in ["adapter_config.json", "adapter_model.safetensors",
                   "adapter_model.bin"]:
    _apath = OUTPUT_DIR / _artifact
    if _apath.exists():
        _apath.unlink()
        print(f"  Removed stale PEFT artifact: {_artifact}")

# ── 5. Reload verification ────────────────────────────────────
print("\nVerifying saved model loads cleanly ...")
try:
    _verify_model = SentenceTransformer(str(OUTPUT_DIR), trust_remote_code=True)
    _test_emb = _verify_model.encode(
        ["Test construction BOQ verification."],
        show_progress_bar=False)
    if np.isnan(_test_emb).any() or np.allclose(_test_emb, 0):
        print("  ⚠️  WARNING: Verification embedding is NaN or zero!")
    else:
        print(f"  ✓ Verified: embedding dim={_test_emb.shape[-1]}, non-zero, no NaN")
    del _verify_model
except Exception as e:
    print(f"  ❌ Verification FAILED: {e}")
    raise

# ── 6. Save training metadata ─────────────────────────────────
n_clusters_actual = len(set(doc_types.values()))
meta = {
    "base_model": MODEL_NAME,
    "n_train": len(train_ds),
    "n_clusters": n_clusters_actual,
    "query_instruction": QUERY_INSTRUCTION,
    "lora_merged": True,
    "version": "v36"
}
(OUTPUT_DIR / "training_meta.json").write_text(json.dumps(meta, indent=2))

gc.collect(); torch.cuda.empty_cache()
print("\n✓ Model saved and ready for retrieval.")


In [ ]:
# ============================================================
# CELL 11 · Build Corpus Embeddings + Retrieval Pipeline
#
# FIX: restore doc_keywords from cluster_state (was missing in old code)
# FIX: QUERY_INSTRUCTION added to query side at inference
#      Corpus docs are embedded WITHOUT prefix (document side)
# ============================================================

# ── Load fine-tuned model ────────────────────────────────────
print(f"Loading fine-tuned model from {OUTPUT_DIR} ...")
best_model = SentenceTransformer(str(OUTPUT_DIR), trust_remote_code=True)
best_model.max_seq_length = MAX_SEQ_LEN
print("  ✓ Loaded (no LoRA → no UNEXPECTED/MISSING key warnings)")
if getattr(best_model[1], "pooling_mode_lasttoken", False) != True:
    print("  ⚠️ WARNING: pooling mode may not be last-token")

# ── Restore cluster state if needed ─────────────────────────
def _load_cluster_state_if_needed():
    required = ["doc_anchors", "doc_types", "doc_display_types",
                "doc_keywords", "doc_tfidf", "stems_list"]
    missing  = [r for r in required if r not in globals()]
    if not missing:
        return  # all present from Cell 7

    print("\nRestoring cluster state from Drive ...")
    state_path = CLUSTER_STATE_DIR / "cluster_state.json"
    tfidf_path = CLUSTER_STATE_DIR / "doc_tfidf.pkl"
    if not state_path.exists():
        raise FileNotFoundError(
            f"cluster_state.json not found at {state_path}.\n"
            f"Run Cell 7 first, or copy from Drive.")

    _st = json.loads(state_path.read_text())
    globals().update({
        "stems_list":              _st["stems_list"],
        "doc_types":               _st["doc_types"],
        "doc_anchors":             _st["doc_anchors"],
        "doc_display_types":       _st["doc_display_types"],
        "doc_keywords":            _st.get("doc_keywords", {}),    # FIX: restored
        "n_clusters_actual":       _st["n_clusters_actual"],
        "cluster_members":         _st["cluster_members"],
        "cluster_names":           {int(k): v
                                    for k,v in _st["cluster_names"].items()},
        "cluster_name_to_doctype": _st.get("cluster_name_to_doctype", {}),
        "test_docs":               _st.get("test_docs", []),
    })
    if tfidf_path.exists():
        with open(tfidf_path, "rb") as f:
            globals()["doc_tfidf"] = pickle.load(f)
    print(f"  ✓ Restored: {len(_st['stems_list'])} docs, "
          f"{_st['n_clusters_actual']} clusters.")

_load_cluster_state_if_needed()

# ── Build corpus embeddings ───────────────────────────────────
# IMPORTANT: corpus docs embedded WITHOUT query instruction prefix
# Only the QUERY side gets the instruction (at retrieval time)
corpus_stems   = [s for s in doc_anchors if doc_types.get(s) != "Cluster_Other"]
corpus_texts   = [doc_anchors[s] for s in corpus_stems]
corpus_types   = [doc_types[s]                 for s in corpus_stems]
corpus_display = [doc_display_types.get(s, s)  for s in corpus_stems]

print(f"\nEncoding {len(corpus_stems)} corpus documents (NO instruction prefix) ...")
corpus_embeddings = best_model.encode(
    corpus_texts,
    batch_size           = 16,
    show_progress_bar    = True,
    convert_to_tensor    = True,
    normalize_embeddings = True,
)

torch.save({
    "embeddings":    corpus_embeddings.cpu(),
    "stems":         corpus_stems,
    "types":         corpus_types,
    "display_types": corpus_display,
    "anchors":       corpus_texts,
}, str(EMBED_CACHE))
print(f"\nCached → {EMBED_CACHE}  |  shape: {tuple(corpus_embeddings.shape)}")


# ── Retrieval function ────────────────────────────────────────
def retrieve_similar_boqs(
    query_pdf_path: str,
    top_k: int = 10,
    exclude_self: bool = True,
    verbose: bool = True,
) -> List[Dict]:
    """
    Upload a BOQ PDF → get top-K most similar BOQs from the corpus.

    Query side  : QUERY_INSTRUCTION + extracted anchor text  (asymmetric)
    Corpus side : plain anchor text (no prefix)
    """
    query_stem = Path(query_pdf_path).stem

    # Check cache first
    if query_stem in doc_anchors:
        anchor = doc_anchors[query_stem]
        if verbose:
            print(f"  Using cached anchor for: {query_stem}")
    else:
        if verbose: print(f"  Extracting text from: {Path(query_pdf_path).name}")
        raw_text = smart_extract(query_pdf_path, verbose=verbose)
        descs    = extract_boq_descriptions(raw_text)
        anchor   = make_smart_anchor(query_stem, descs)

    if not anchor.strip():
        print("⚠️  WARNING: Could not extract usable text from query PDF.")
        return []

    # FIX: prepend instruction for the query side
    query_text = QUERY_INSTRUCTION + anchor

    q_emb  = best_model.encode(
        [query_text], convert_to_tensor=True, normalize_embeddings=True,
        show_progress_bar=False)
    scores = st_util.cos_sim(q_emb, corpus_embeddings.to(q_emb.device))[0]

    results = []
    for idx in scores.argsort(descending=True):
        stem = corpus_stems[idx.item()]
        if exclude_self and stem == query_stem:
            continue
        results.append({
            "rank":         len(results) + 1,
            "stem":         stem,
            "cluster":      corpus_types[idx.item()],
            "display_type": corpus_display[idx.item()],
            "score":        float(scores[idx]),
        })
        if len(results) >= top_k:
            break
    return results


# ── Demo ─────────────────────────────────────────────────────
print("\n" + "="*70)
print("DEMO — Retrieving similar BOQs for first corpus document")
print("="*70)

demo_pdf  = str(pdf_files[0])
demo_stem = pdf_files[0].stem
print(f"Query   : {pdf_files[0].name}")
print(f"Cluster : {doc_types.get(demo_stem, 'N/A')}")
print(f"Display : {doc_display_types.get(demo_stem, 'N/A')}")

results = retrieve_similar_boqs(demo_pdf, top_k=10, verbose=True)
print(f"\n{'Rank':<5} {'Score':<8} {'Cluster':<45} Document")
print("-"*100)
for r in results:
    print(f"  {r['rank']:<3}  {r['score']:.4f}  {r['cluster']:<45}  {r['stem']}")


In [ ]:
# ============================================================
# CELL 12 · Upload a NEW BOQ PDF and Retrieve Similar Ones
# ============================================================
from google.colab import files as colab_files

# ── Lazy-load model if needed ─────────────────────────────────
if "best_model" not in globals() or best_model is None:
    print("Reloading fine-tuned model from Drive ...")
    best_model = SentenceTransformer(str(OUTPUT_DIR), trust_remote_code=True)
    best_model.max_seq_length = MAX_SEQ_LEN
    print(f"  Loaded from {OUTPUT_DIR}")

if "corpus_embeddings" not in globals() or corpus_embeddings is None:
    print("Reloading corpus embeddings from Drive ...")
    _cache = torch.load(str(EMBED_CACHE))
    corpus_embeddings = _cache["embeddings"].to(DEVICE)
    corpus_stems      = _cache["stems"]
    corpus_types      = _cache["types"]
    corpus_display    = _cache.get("display_types", corpus_types)
    print(f"  Reloaded {len(corpus_stems):,} documents.")

_load_cluster_state_if_needed()  # restores doc_tfidf, doc_anchors, etc.

# ── Upload + Retrieve ─────────────────────────────────────────
print("\nUpload a BOQ PDF to find the 10 most similar BOQs in the database ...")
new_uploaded = colab_files.upload()

if new_uploaded:
    new_name = list(new_uploaded.keys())[0]
    new_path = f"/content/{new_name}"
    print(f"\nQuery file: {new_name}")
    print("─"*80)

    results = retrieve_similar_boqs(
        new_path, top_k=10, exclude_self=False, verbose=True)

    if results:
        print(f"\n{'Rank':<5} {'Score':<8} {'Cluster':<40} {'Display':<30} Document")
        print("─"*120)
        for r in results:
            marker = "  ← (self)" if r["stem"] == Path(new_name).stem else ""
            print(f"  {r['rank']:<3}  {r['score']:.4f}  "
                  f"{r['cluster'][:39]:<40}  "
                  f"{r['display_type'][:28]:<30}  "
                  f"{r['stem']}{marker}")
    else:
        print("No results returned — check if anchor extraction succeeded.")
else:
    print("No file uploaded.")


In [ ]:
# ============================================================
# CELL EVAL-A · Corpus-Wide Retrieval Benchmark
#
# FIX: doc_keywords now properly restored from cluster_state
# FIX: instruction prefix applied to query side only in eval
# Ground truth: cluster assignments (content-based) = relevance labels
# ============================================================
import time
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Restore cluster state ────────────────────────────────────
_CLUSTER_STATE_DIR = CLUSTER_STATE_DIR
_state_loaded = False

if (_CLUSTER_STATE_DIR / "cluster_state.json").exists():
    print("Loading persisted cluster state ...")
    try:
        _st  = json.loads((_CLUSTER_STATE_DIR / "cluster_state.json").read_text())
        stems_list        = _st["stems_list"]
        doc_types         = _st["doc_types"]
        doc_anchors       = _st["doc_anchors"]
        doc_display_types = _st["doc_display_types"]
        doc_keywords      = _st.get("doc_keywords", {})   # FIX: restored
        n_clusters_actual = _st["n_clusters_actual"]
        cluster_members   = _st["cluster_members"]
        cluster_names     = {int(k):v for k,v in _st["cluster_names"].items()}
        test_docs_set     = set(_st.get("test_docs", []))
        with open(_CLUSTER_STATE_DIR / "doc_tfidf.pkl", "rb") as f:
            doc_tfidf = pickle.load(f)
        _state_loaded = True
        print(f"  ✓ {len(stems_list)} docs, {n_clusters_actual} clusters")
    except Exception as e:
        print(f"  ⚠️  Load failed ({e}) — falling back to in-memory globals")

if not _state_loaded:
    for _req in ["doc_types","doc_anchors","stems_list","doc_tfidf"]:
        if _req not in globals():
            raise RuntimeError(f"'{_req}' not in globals. Run Cell 7 first.")
    test_docs_set = set(test_docs) if "test_docs" in globals() else set()

# ── Exclude Cluster_Other from evaluation ────────────────────
_other_stems = {s for s,t in doc_types.items() if t == "Cluster_Other"}
if _other_stems:
    print(f"  Excluding {len(_other_stems)} Cluster_Other docs from eval")

corpus_stems_eval   = [s for s in doc_anchors if s not in _other_stems]
corpus_texts_eval   = [doc_anchors[s]             for s in corpus_stems_eval]
query_texts_eval    = [QUERY_INSTRUCTION + doc_anchors[s] for s in corpus_stems_eval]
query_texts_eval    = [QUERY_INSTRUCTION + doc_anchors[s] for s in corpus_stems_eval]
corpus_types_eval   = [doc_types[s]               for s in corpus_stems_eval]
is_test_arr         = np.array([s in test_docs_set for s in corpus_stems_eval])
n_eval              = len(corpus_stems_eval)
print(f"  {n_eval} docs  ·  {len(set(corpus_types_eval))} clusters")

# ── Safe encode helper ────────────────────────────────────────
def encode_safe(mdl, texts, start_batch: int = 16) -> np.ndarray:
    batch = start_batch
    while batch >= 1:
        try:
            embs = mdl.encode(texts, batch_size=batch, show_progress_bar=True,
                               convert_to_tensor=True, normalize_embeddings=True)
            out  = embs.cpu().float().numpy()
            del embs; torch.cuda.empty_cache()
            return out
        except torch.cuda.OutOfMemoryError:
            print(f"   OOM at batch={batch} → halving ...")
            torch.cuda.empty_cache(); gc.collect()
            batch //= 2
    return mdl.encode(texts, batch_size=2, show_progress_bar=True,
                      convert_to_tensor=False, normalize_embeddings=True,
                      device="cpu")

# ── Encode corpus with fine-tuned model ──────────────────────
# FIX: corpus (document side) has NO instruction prefix
t0 = time.time()
print("\nEncoding with fine-tuned model (document side, no prefix) ...")
_ft = SentenceTransformer(str(OUTPUT_DIR), trust_remote_code=True)
_ft.max_seq_length = MAX_SEQ_LEN
ft_np = encode_safe(_ft, corpus_texts_eval)
del _ft; gc.collect(); torch.cuda.empty_cache()
print(f"  {time.time()-t0:.1f}s | shape: {ft_np.shape}")

# ── Encode with base model (cached) ──────────────────────────
BASE_EMBED_CACHE = Path(f"{DRIVE_BASE}/base_embeddings.pt")
base_np_corpus = None
base_np_queries = None
if BASE_EMBED_CACHE.exists():
    try:
        _bc = torch.load(str(BASE_EMBED_CACHE))
        _bc_stems = _bc.get("stems", [])
        if set(_bc_stems) == set(corpus_stems_eval) and "queries" in _bc and _bc.get("source") == "anchor":
            _idx = {s:i for i,s in enumerate(_bc_stems)}
            base_np_corpus = _bc["embeddings"].numpy()[[_idx[s] for s in corpus_stems_eval]]
            base_np_queries = _bc["queries"].numpy()[[_idx[s] for s in corpus_stems_eval]]
            print(f"  Loaded cached base embeddings.")
        else:
            base_np_corpus = None
    except Exception as e:
        print(f"  Base cache stale ({e}), re-encoding ...")
        base_np_corpus = None

if base_np_corpus is None:
    print("\nEncoding with BASE Qwen3 model (no fine-tuning) ...")
    _base = SentenceTransformer(MODEL_NAME, trust_remote_code=True)
    _base.max_seq_length = MAX_SEQ_LEN
    # Set last-token pooling for base model
    try:
        _base[1].pooling_mode_lasttoken   = True
        _base[1].pooling_mode_mean_tokens = False
    except Exception: pass
    base_np_corpus = encode_safe(_base, corpus_texts_eval)
    base_np_queries = encode_safe(_base, query_texts_eval)
    del _base; gc.collect(); torch.cuda.empty_cache()
    try:
        torch.save({"embeddings": torch.tensor(base_np_corpus), "queries": torch.tensor(base_np_queries), "stems": corpus_stems_eval},
                   str(BASE_EMBED_CACHE))
        print(f"  Cached base embeddings → {BASE_EMBED_CACHE}")
    except Exception as e:
        print(f"  WARNING: Could not cache: {e}")

# ══════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════
def compute_retrieval_metrics(emb, types, is_test, ks=(1,3,5,10)):
    types_arr = np.array(types)
    n         = len(types_arr)
    S         = emb @ emb.T
    np.fill_diagonal(S, -1e9)
    test_idx  = np.where(is_test)[0]
    if len(test_idx) == 0:
        return {"error": "No test docs"}

    S_test     = S[test_idx]
    types_test = types_arr[test_idx]
    R = (types_test[:,None] == types_arr[None,:]).astype(np.float32)
    for i,idx in enumerate(test_idx): R[i,idx] = 0.0

    n_rel   = R.sum(axis=1)
    valid   = n_rel > 0
    ranked  = np.argsort(-S_test, axis=1)
    rel_r   = R[np.arange(len(test_idx))[:,None], ranked]
    first_r = np.where(rel_r, np.arange(1,n+1), n+1).min(axis=1)
    mrr     = float(np.mean(1.0/first_r[valid])) if valid.any() else 0.0
    cr      = np.cumsum(rel_r, axis=1)
    ap      = (cr/np.arange(1,n+1)*rel_r).sum(1)/np.maximum(n_rel,1)
    map_    = float(np.mean(ap[valid])) if valid.any() else 0.0
    log2d   = np.log2(np.arange(2,n+2))
    res     = {"MRR":mrr,"MAP":map_}
    for k in ks:
        rk = rel_r[:,:k]
        rr = rk.sum(1)
        p  = rr/k; r=rr/np.maximum(n_rel,1)
        f1 = np.where((p+r)>0, 2*p*r/(p+r), 0.0)
        dcg  = (rk/log2d[:k]).sum(1)
        idl  = np.array([(1/np.log2(np.arange(2,min(k,int(nr))+2))).sum()
                          if nr>0 else 0.0 for nr in n_rel])
        ndcg = np.where(idl>0, dcg/idl, 0.0)
        res.update({
            f"Hit@{k}":  float(np.mean((rr>0)[valid])) if valid.any() else 0.0,
            f"P@{k}":    float(np.mean(p[valid]))       if valid.any() else 0.0,
            f"R@{k}":    float(np.mean(r[valid]))       if valid.any() else 0.0,
            f"F1@{k}":   float(np.mean(f1[valid]))      if valid.any() else 0.0,
            f"NDCG@{k}": float(np.mean(ndcg[valid]))    if valid.any() else 0.0,
        })
    res["n_queries"] = int(valid.sum())
    res["n_clusters"]= int(len(set(types_test[valid])))
    return res

def compute_quality(emb, types):
    ta = np.array(types)
    S  = emb @ emb.T
    mi = ta[:,None]==ta[None,:]; mo=~mi
    np.fill_diagonal(mi,False); np.fill_diagonal(mo,False)
    intra = float(S[mi].mean()) if mi.any() else float("nan")
    inter = float(S[mo].mean()) if mo.any() else float("nan")
    lmap  = {t:i for i,t in enumerate(set(types))}
    try: sil = float(silhouette_score(emb, [lmap[t] for t in types], metric="cosine"))
    except: sil = float("nan")
    return {"Intra-class sim":intra,"Inter-class sim":inter,
            "Sim gap (Δ)":intra-inter,"Silhouette":sil}

def compute_retrieval_metrics_asym(emb_q, emb_c, types, is_test, ks=(1,3,5,10)):
    types_arr = np.array(types)
    n         = len(types_arr)
    S         = emb_q @ emb_c.T
    np.fill_diagonal(S, -1e9)
    test_idx  = np.where(is_test)[0]
    if len(test_idx) == 0:
        return {"error": "No test docs"}

    S_test     = S[test_idx]
    types_test = types_arr[test_idx]
    R = (types_test[:,None] == types_arr[None,:]).astype(np.float32)
    for i,idx in enumerate(test_idx): R[i,idx] = 0.0

    n_rel   = R.sum(axis=1)
    valid   = n_rel > 0
    ranked  = np.argsort(-S_test, axis=1)
    rel_r   = R[np.arange(len(test_idx))[:,None], ranked]
    first_r = np.where(rel_r, np.arange(1,n+1), n+1).min(axis=1)
    mrr     = float(np.mean(1.0/first_r[valid])) if valid.any() else 0.0
    cr      = np.cumsum(rel_r, axis=1)
    ap      = (cr/np.arange(1,n+1)*rel_r).sum(1)/np.maximum(n_rel,1)
    map_    = float(np.mean(ap[valid])) if valid.any() else 0.0
    log2d   = np.log2(np.arange(2,n+2))
    res     = {"MRR":mrr,"MAP":map_}
    for k in ks:
        rk = rel_r[:,:k]
        rr = rk.sum(1)
        p  = rr/k; r=rr/np.maximum(n_rel,1)
        f1 = np.where((p+r)>0, 2*p*r/(p+r), 0.0)
        dcg  = (rk/log2d[:k]).sum(1)
        idl  = np.array([(1/np.log2(np.arange(2,min(k,int(nr))+2))).sum()
                          if nr>0 else 0.0 for nr in n_rel])
        ndcg = np.where(idl>0, dcg/idl, 0.0)
        res.update({
            f"Hit@{k}":  float(np.mean((rr>0)[valid])) if valid.any() else 0.0,
            f"P@{k}":    float(np.mean(p[valid]))       if valid.any() else 0.0,
            f"R@{k}":    float(np.mean(r[valid]))       if valid.any() else 0.0,
            f"F1@{k}":   float(np.mean(f1[valid]))      if valid.any() else 0.0,
            f"NDCG@{k}": float(np.mean(ndcg[valid]))    if valid.any() else 0.0,
        })
    res["n_queries"] = int(valid.sum())
    res["n_clusters"]= int(len(set(types_test[valid])))
    return res

print("\nComputing metrics ...")
ft_ret   = compute_retrieval_metrics_asym(ft_np_queries, ft_np_corpus, corpus_types_eval, is_test_arr)
base_ret = compute_retrieval_metrics_asym(base_np_queries, base_np_corpus, corpus_types_eval, is_test_arr)
ft_qual  = compute_quality(ft_np_corpus,   corpus_types_eval)
base_qual= compute_quality(base_np_corpus, corpus_types_eval)
print("  ✓ Done")

# ── HTML Report ───────────────────────────────────────────────
def _dh(a,b):
    if not (isinstance(a,float) and isinstance(b,float)): return "—"
    d=a-b; c="#22c55e" if d>0 else "#f87171"; s="▲" if d>0 else "▼"
    return f"<span style='color:{c};font-weight:700'>{s}{abs(d):.4f}</span>"

def _vh(v):
    if not isinstance(v,float):
        return f"<td style='padding:5px 10px;color:#e2e8f0'>{v}</td>"
    c="#22c55e" if v>=.8 else "#facc15" if v>=.5 else "#f87171"
    return f"<td style='padding:5px 10px;color:{c};font-weight:600'>{v:.4f}</td>"

def tbl(rows,title,cols):
    hdr="".join(f"<th style='padding:5px 10px;color:#94a3b8;font-size:11px;"
                f"border-bottom:2px solid #334155'>{c}</th>" for c in cols)
    bdy=""
    for r in rows:
        cells="".join(
            _vh(v) if i>0 and isinstance(v,float) else
            f"<td style='padding:5px 10px'>{v}</td>"
            for i,v in enumerate(r))
        bdy+=f"<tr>{cells}</tr>"
    return (f"<div style='margin:6px 0;background:#0f172a;border-radius:8px;"
            f"border:1px solid #1e293b'>"
            f"<div style='padding:8px 14px;background:#1e293b;color:#e2e8f0;"
            f"font-size:13px;font-weight:600'>{title}</div>"
            f"<table style='width:100%;border-collapse:collapse'>"
            f"<thead><tr style='background:#162032'>{hdr}</tr></thead>"
            f"<tbody>{bdy}</tbody></table></div>")

metrics_to_show = ["MRR","MAP","Hit@1","Hit@3","Hit@5","Hit@10",
                   "P@1","P@3","P@5","P@10",
                   "R@1","R@3","R@5","R@10",
                   "F1@1","F1@3","F1@5","F1@10",
                   "NDCG@1","NDCG@5","NDCG@10"]
qual_metrics = ["Intra-class sim","Inter-class sim","Sim gap (Δ)","Silhouette"]

ret_rows  = [[m, ft_ret.get(m,float("nan")), base_ret.get(m,float("nan")),
              _dh(ft_ret.get(m), base_ret.get(m))] for m in metrics_to_show]
qual_rows = [[m, ft_qual.get(m,float("nan")), base_qual.get(m,float("nan")),
              _dh(ft_qual.get(m), base_qual.get(m))] for m in qual_metrics]

display(HTML(f"""
<div style='font-family:monospace;color:#e2e8f0;padding:4px'>
  <div style='font-size:15px;font-weight:700;color:#38bdf8;margin-bottom:6px'>
    📊 BOQ Eval · {n_eval} docs · {ft_ret.get('n_clusters','?')} clusters ·
    {ft_ret.get('n_queries','?')} test queries · {time.time()-t0:.0f}s
  </div>
  <div style='font-size:11px;color:#64748b;margin-bottom:8px'>
    Anchor: TF-IDF sum-scored Smart Anchors ·
    Query side: instruction-prefixed · Corpus side: plain ·
    GT: content-based cluster labels
  </div>
  {tbl(ret_rows,  "🎯 Retrieval Metrics",   ["Metric","Fine-Tuned","Base Qwen3","Δ"])}
  {tbl(qual_rows, "🔬 Embedding Quality",   ["Metric","Fine-Tuned","Base Qwen3","Δ"])}
</div>
"""))

try:
    p = Path(DRIVE_BASE) / "eval_metrics.json"
    p.write_text(json.dumps(
        {"finetuned":{**ft_ret,**ft_qual},
         "base":{**base_ret,**base_qual},
         "n_docs":n_eval},
        indent=2, default=str))
    print(f"\n✓ Metrics saved → {p}")
except Exception as e:
    print(f"  WARNING: Could not save: {e}")
